# 06 — Final test: how much could ANY corruption-aware adaptive sigma gain?
Oracle experiment on the 15 standard corruptions (sev 5), BN-only TENT, lr 1e-3, bs 64, W4A4 + W3A3, 3 seeds.
Compares: STE, best single fixed sigma/Delta, and a leave-one-seed-out "oracle" that picks sigma per corruption on one seed and is scored on the other two (removes winner's-curse optimism).
**Decision rule:** build an adaptive rule only if (oracle - best fixed) has 95% CI lower bound > 0.3 pts AND best sigma clusters by corruption type.

In [ ]:
import sys, os, glob, copy
import numpy as np, pandas as pd, torch, torch.nn as nn
sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/**/qtta_lib.py", recursive=True)[0]))
import qtta_lib as Q
from qtta_lib import QCFG
dev = "cuda" if torch.cuda.is_available() else "cpu"

CATS = {"noise": ["gaussian_noise", "shot_noise", "impulse_noise"],
        "blur": ["defocus_blur", "glass_blur", "motion_blur", "zoom_blur"],
        "weather": ["snow", "frost", "fog", "brightness"],
        "digital": ["contrast", "elastic_transform", "pixelate", "jpeg_compression"]}
CORR = [c for v in CATS.values() for c in v]
CAT_OF = {c: k for k, v in CATS.items() for c in v}
SEEDS, SEV = [0, 1, 2], 5
METHODS = [("STE", "ste", 0.0)] + [(f"g{s}", "gauss", s) for s in [0.25, 0.35, 0.5, 1.0, 2.0]]
MEAN = torch.tensor(Q.MEAN).view(1, 3, 1, 1); STD = torch.tensor(Q.STD).view(1, 3, 1, 1)

def load_c(name, sev=SEV):
    path = glob.glob(f"/kaggle/input/**/{name}.npy", recursive=True)[0]
    lab = os.path.join(os.path.dirname(path), "labels.npy")
    sl = slice((sev - 1) * 10000, sev * 10000)
    x = torch.from_numpy(np.array(np.load(path, mmap_mode="r")[sl])).permute(0, 3, 1, 2).float() / 255
    return ((x - MEAN) / STD).to(dev), torch.from_numpy(np.load(lab)[sl]).long().to(dev)

fp, _ = Q.load_fp32(Q.find_file("resnet20_fp32.pt"), dev)
train_loader, _ = Q.get_cifar10(bs=500)
MODELS = {n: Q.quantize_model(fp, wb, ab, train_loader, dev)[0] for n, (wb, ab) in {"W4A4": (4, 4), "W3A3": (3, 3)}.items()}

def entropy(lg): return -(lg.softmax(1) * lg.log_softmax(1)).sum(1).mean()

def run_tta(model, x, y, mode, sigma, lr=1e-3, bs=64, seed=0):
    m = copy.deepcopy(model); m.train()
    for p in m.parameters(): p.requires_grad_(False)
    params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.track_running_stats = False; mod.running_mean = None; mod.running_var = None
            mod.weight.requires_grad_(True); mod.bias.requires_grad_(True); params += [mod.weight, mod.bias]
    QCFG.mode, QCFG.sigma = mode, sigma
    opt = torch.optim.Adam(params, lr=lr)
    perm = torch.randperm(len(x), generator=torch.Generator().manual_seed(seed)).to(dev)
    correct = 0
    for i in range(0, len(x), bs):
        idx = perm[i:i + bs]
        logits = m(x[idx])
        opt.zero_grad(set_to_none=True); entropy(logits).backward(); opt.step()
        correct += (logits.argmax(1) == y[idx]).sum().item()
    return 100 * correct / len(x)

## Run (saves after every corruption; ~1 h on a T4)

In [ ]:
res = []
for c in CORR:
    x, y = load_c(c)
    for mn, base in MODELS.items():
        for name, mode, s in METHODS:
            for seed in SEEDS:
                res.append(dict(model=mn, corruption=c, method=name, seed=seed, acc=run_tta(base, x, y, mode, s, seed=seed)))
    pd.DataFrame(res).to_csv("/kaggle/working/oracle_15c.csv", index=False)
    print("done", c, flush=True)
    del x, y; torch.cuda.empty_cache()

## Analysis

In [ ]:
df = pd.DataFrame(res)
G = [n for n, _, _ in METHODS if n != "STE"]
rng = np.random.default_rng(0)
def boot(v, B=5000):
    v = np.asarray(v); b = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(B)]
    return v.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

for mn in MODELS:
    A = df[df.model == mn].pivot_table(index=["corruption", "seed"], columns="method", values="acc")
    r_or, r_fix, r_ste = {c: [] for c in CORR}, {c: [] for c in CORR}, {c: [] for c in CORR}
    for s in SEEDS:                                   # leave-one-seed-out selection
        sel = A.xs(s, level="seed"); ev = A.drop(s, level="seed").groupby("corruption").mean()
        fixed = sel[G].mean().idxmax()
        for c in CORR:
            r_or[c].append(ev.loc[c, sel.loc[c, G].idxmax()]); r_fix[c].append(ev.loc[c, fixed]); r_ste[c].append(ev.loc[c, "STE"])
    f = lambda d: np.array([np.mean(d[c]) for c in CORR])
    gap_or_fix, gap_fix_ste = f(r_or) - f(r_fix), f(r_fix) - f(r_ste)
    M = A.groupby("corruption").mean()
    naive = M[G].max(axis=1).mean() - M[G].mean().max()
    print(f"=== {mn} ===  mean acc: STE {M['STE'].mean():.2f} | best fixed ({M[G].mean().idxmax()}) {M[G].mean().max():.2f}")
    print("best fixed - STE        : mean %.2f  95%% CI [%.2f, %.2f]" % boot(gap_fix_ste))
    print("oracle - best fixed (CV): mean %.2f  95%% CI [%.2f, %.2f]" % boot(gap_or_fix))
    print(f"oracle - best fixed (naive, optimistic): {naive:.2f}")
    best = M[G].idxmax(axis=1)
    print("best sigma by category:")
    print(pd.crosstab(best.map(CAT_OF), best).to_string())
    print()
print(df.pivot_table(index="corruption", columns=["model", "method"], values="acc").round(2).to_string())